# Chains in LangChain

## Outline

* LLMChain
* Sequential Chains
  * SimpleSequentialChain
  * SequentialChain
* Router Chain

Chain（链）是 LangChain 最核心的抽象：把"prompt 模板 + LLM（+ 输出解析）"组合成一个可复用的单元，
并且可以把多个 chain 首尾相连、组合成更复杂的流程。

> 注：本 notebook 里 `LLMChain`、`SimpleSequentialChain`、`SequentialChain`、`MultiPromptChain` 等
> 都是"经典链"写法，在新版 langchain 中已挪到 `langchain_classic` 包并标记为 deprecated
> （官方推荐用 `prompt | llm` 这种 LCEL 管道写法代替），但目前仍可正常运行。
> 另外，本 notebook 用到的 `Data.csv` 数据文件在当前工作目录下不存在，涉及读取该文件的 cell 无法在本地跑通，
> 这属于数据文件缺失，不是代码逻辑问题，这里不做修复。

In [ ]:
# 屏蔽掉不影响运行结果的警告信息（比如 LangChainDeprecationWarning），让输出更干净
import  warnings
warnings.filterwarnings('ignore')

In [ ]:
# 加载 .env 中的环境变量（如 OPENAI_API_KEY）
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

In [ ]:
# account for deprecation of LLM model
# 根据当前日期判断该用哪个 gpt-3.5-turbo 版本名，逻辑本身没问题（详见 L1/L2 的说明）
import datetime
# Get the current date
current_date = datetime.datetime.now().date()

# Define the date after which the model should be set to "gpt-3.5-turbo"
target_date = datetime.date(2024, 6, 12)

# Set the model variable based on the current date
if current_date > target_date:
    llm_model = "gpt-3.5-turbo"
else:
    llm_model = "gpt-3.5-turbo-0301"

In [ ]:
# 读取一份产品数据 CSV，后面会用某一列（Review）作为 chain 的输入
# 【环境限制】Data.csv 在当前目录下不存在（课程平台提供，未随 notebook 一起分发），
# 这行会报 FileNotFoundError；这是数据缺失问题，不是代码逻辑 bug，这里不做修复
import pandas as pd
df = pd.read_csv('Data.csv')

In [ ]:
# 预览前几行数据，确认列名（应该有 Product、Review 等列）
df.head()

In [ ]:
# 【版本兼容性修复】原代码 from langchain.chat_models import ChatOpenAI 等三行在当前 langchain 1.4.0 下
# 全部会报错：langchain.chat_models 里已没有 ChatOpenAI，langchain.prompts / langchain.chains 模块已不存在。
# 正确路径：
#   ChatOpenAI       -> langchain_openai（独立子包）
#   ChatPromptTemplate -> langchain_core.prompts（轻量核心包）
#   LLMChain          -> langchain_classic.chains（经典链，已 deprecated 但可用）
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains import LLMChain

In [ ]:
# temperature=0.9：调高随机性，鼓励模型给出更有创意的公司名称
llm = ChatOpenAI(temperature=0.9, model=llm_model)

In [ ]:
# 定义一个带 {product} 占位符的 prompt 模板：让模型给某种产品的公司起名
prompt = ChatPromptTemplate.from_template(
    "What is the best name to describe \
    a company that makes {product}?"
)

In [ ]:
# LLMChain 是最基础的 Chain：把 llm 和 prompt 绑定在一起，调用时只需要传入模板变量
chain = LLMChain(llm=llm, prompt=prompt)

In [ ]:
# 【提示】chain.run(product) 是旧式调用方法，当前版本仍可用但已 deprecated，
# 新写法推荐 chain.invoke({"product": product})。这里保留 .run() 贴近原课程写法。
product = "Queen Size Sheet Set"
chain.run(product)

In [ ]:
# 【版本兼容性修复】同上，正确路径是 langchain_classic.chains
# SimpleSequentialChain：把多个 chain 串起来，上一个 chain 的唯一输出自动作为下一个 chain 的唯一输入
from langchain_classic.chains import SimpleSequentialChain

In [ ]:
llm = ChatOpenAI(temperature=0.9, model=llm_model)

# prompt template 1
# 第一步：根据产品类型，起一个公司名
first_prompt = ChatPromptTemplate.from_template(
    "What is the best name to describe \
    a company that makes {product}?"
)

# Chain 1
chain_one = LLMChain(llm=llm, prompt=first_prompt)

In [ ]:
# prompt template 2
# 第二步：根据第一步生成的公司名，写一段 20 词的公司简介
second_prompt = ChatPromptTemplate.from_template(
    "Write a 20 words description for the following \
    company:{company_name}"
)
# chain 2
chain_two = LLMChain(llm=llm, prompt=second_prompt)

In [ ]:
# SimpleSequentialChain 要求每个子 chain 只有一个输入、一个输出，
# 上一个 chain 的输出会原样作为下一个 chain 的输入（这里是 chain_one 的公司名 -> chain_two 的 company_name）
overall_simple_chain = SimpleSequentialChain(chains=[chain_one, chain_two],
                                             verbose=True
                                            )

In [ ]:
overall_simple_chain.run(product)

In [ ]:
# SequentialChain 比 SimpleSequentialChain 更灵活：支持多输入多输出，
# 需要显式声明 input_variables / output_variables，并给每个子 chain 指定 output_key
from langchain_classic.chains import SequentialChain

In [ ]:
llm = ChatOpenAI(temperature=0.9, model=llm_model)

# prompt template 1: translate to english
first_prompt = ChatPromptTemplate.from_template(
    "Translate the following review to english:"
    "\n\n{Review}"
)
# chain 1: input= Review and output= English_Review
# output_key 指定这个 chain 的输出在整体流程里叫什么名字，后续 chain 可以通过这个 key 引用它
chain_one = LLMChain(llm=llm, prompt=first_prompt,
                     output_key="English_Review"
                    )


In [ ]:
second_prompt = ChatPromptTemplate.from_template(
    "Can you summarize the following review in 1 sentence:"
    "\n\n{English_Review}"
)
# chain 2: input= English_Review and output= summary
# 这里的输入变量名 {English_Review} 正好对应 chain_one 的 output_key，实现链式传递
chain_two = LLMChain(llm=llm, prompt=second_prompt,
                     output_key="summary"
                    )


In [ ]:
# prompt template 3: translate to english
# 第三条链是独立分支：直接用原始 Review（不是翻译后的）判断这段评论是什么语言
third_prompt = ChatPromptTemplate.from_template(
    "What language is the following review:\n\n{Review}"
)
# chain 3: input= Review and output= language
chain_three = LLMChain(llm=llm, prompt=third_prompt,
                       output_key="language"
                      )


In [ ]:

# prompt template 4: follow up message
# 第四条链依赖前面两条链的输出（summary 和 language），生成用对应语言写的回复
fourth_prompt = ChatPromptTemplate.from_template(
    "Write a follow up response to the following "
    "summary in the specified language:"
    "\n\nSummary: {summary}\n\nLanguage: {language}"
)
# chain 4: input= summary, language and output= followup_message
chain_four = LLMChain(llm=llm, prompt=fourth_prompt,
                      output_key="followup_message"
                     )


In [ ]:
# overall_chain: input= Review
# and output= English_Review,summary, followup_message
# input_variables 声明整个流程需要用户提供哪些初始变量；
# output_variables 声明最终要返回哪些中间/最终结果（这里没有把 language 也列进去，只是不作为最终返回值，
# 但它在内部依然会被 chain_four 用到，这是原课程的设计，不是 bug）
overall_chain = SequentialChain(
    chains=[chain_one, chain_two, chain_three, chain_four],
    input_variables=["Review"],
    output_variables=["English_Review", "summary","followup_message"],
    verbose=True
)

In [ ]:
# 取数据集里第 6 行（索引 5）的评论作为输入
# 【提示】overall_chain(review) 是旧式的"把 chain 当函数调用"写法，当前版本仍可用（deprecated），
# 等价于 overall_chain.invoke(review)
review = df.Review[5]
overall_chain(review)

In [ ]:
# 四个"专家角色"的 prompt 模板：分别扮演物理教授、数学家、历史学家、计算机科学家
# 每个模板都包含 {input} 占位符，用来接收用户的原始问题
physics_template = """You are a very smart physics professor. \
You are great at answering questions about physics in a concise\
and easy to understand manner. \
When you don't know the answer to a question you admit\
that you don't know.

Here is a question:
{input}"""


math_template = """You are a very good mathematician. \
You are great at answering math questions. \
You are so good because you are able to break down \
hard problems into their component parts,
answer the component parts, and then put them together\
to answer the broader question.

Here is a question:
{input}"""

history_template = """You are a very good historian. \
You have an excellent knowledge of and understanding of people,\
events and contexts from a range of historical periods. \
You have the ability to think, reflect, debate, discuss and \
evaluate the past. You have a respect for historical evidence\
and the ability to make use of it to support your explanations \
and judgements.

Here is a question:
{input}"""


computerscience_template = """ You are a successful computer scientist.\
You have a passion for creativity, collaboration,\
forward-thinking, confidence, strong problem-solving capabilities,\
understanding of theories and algorithms, and excellent communication \
skills. You are great at answering coding questions. \
You are so good because you know how to solve a problem by \
describing the solution in imperative steps \
that a machine can easily interpret and you know how to \
choose a solution that has a good balance between \
time complexity and space complexity.

Here is a question:
{input}"""

In [ ]:
# 用 name + description + prompt_template 描述每个"目标链"（destination chain），
# description 会被路由链用来判断某个问题该分发给哪个专家
prompt_infos = [
    {
        "name": "physics",
        "description": "Good for answering questions about physics",
        "prompt_template": physics_template
    },
    {
        "name": "math",
        "description": "Good for answering math questions",
        "prompt_template": math_template
    },
    {
        "name": "History",
        "description": "Good for answering history questions",
        "prompt_template": history_template
    },
    {
        "name": "computer science",
        "description": "Good for answering computer science questions",
        "prompt_template": computerscience_template
    }
]

In [ ]:
# 【版本兼容性修复】原代码 from langchain.chains.router / langchain.prompts 在当前版本已不存在，
# MultiPromptChain / LLMRouterChain / RouterOutputParser 属于经典链，正确路径是 langchain_classic.chains.router；
# PromptTemplate 属于核心组件，正确路径是 langchain_core.prompts
from langchain_classic.chains.router import MultiPromptChain
from langchain_classic.chains.router.llm_router import LLMRouterChain,RouterOutputParser
from langchain_core.prompts import PromptTemplate

In [ ]:
# temperature=0：路由决策需要稳定、确定的输出，不需要创意
llm = ChatOpenAI(temperature=0, model=llm_model)

In [ ]:

# 根据 prompt_infos，为每个专家角色构建一个独立的 LLMChain，
# destination_chains 是一个 {name: chain} 的映射表，供路由链根据"目的地名字"分发请求
destination_chains = {}
for p_info in prompt_infos:
    name = p_info["name"]
    prompt_template = p_info["prompt_template"]
    prompt = ChatPromptTemplate.from_template(template=prompt_template)
    chain = LLMChain(llm=llm, prompt=prompt)
    destination_chains[name] = chain

# 把每个目标链的 "name: description" 拼成一段文本，供路由 prompt 展示"有哪些候选"
destinations = [f"{p['name']}: {p['description']}" for p in prompt_infos]
destinations_str = "\n".join(destinations)

In [ ]:
# default_chain：当路由链找不到合适的专家（判定为 "DEFAULT"）时兜底使用的通用链
default_prompt = ChatPromptTemplate.from_template("{input}")
default_chain = LLMChain(llm=llm, prompt=default_prompt)

In [ ]:
# 路由 prompt 模板：让 LLM 判断用户的问题应该交给哪个专家（destination），
# 并且允许它顺带修改一下问题本身（next_inputs）
# 注意：这里用了四层花括号 {{{{ }}}} 是因为这段模板要先经过一次 .format(destinations=...) 处理，
# 而 {{{{ }}}} 在第一次 .format() 后会变成 {{ }}，留给后续 ChatPromptTemplate/PromptTemplate 再解析一次，
# 从而在最终渲染结果里保留字面意义上的一层花括号（用于展示 JSON 结构），这是故意的双重转义，不是笔误
MULTI_PROMPT_ROUTER_TEMPLATE = """Given a raw text input to a \
language model select the model prompt best suited for the input. \
You will be given the names of the available prompts and a \
description of what the prompt is best suited for. \
You may also revise the original input if you think that revising\
it will ultimately lead to a better response from the language model.

<< FORMATTING >>
Return a markdown code snippet with a JSON object formatted to look like:
```json
{{{{
    "destination": string \ "DEFAULT" or name of the prompt to use in {destinations}
    "next_inputs": string \ a potentially modified version of the original input
}}}}
```

REMEMBER: The value of “destination” MUST match one of \
the candidate prompts listed below.\
If “destination” does not fit any of the specified prompts, set it to “DEFAULT.”
REMEMBER: "next_inputs" can just be the original input \
if you don't think any modifications are needed.

<< CANDIDATE PROMPTS >>
{destinations}

<< INPUT >>
{{input}}

<< OUTPUT (remember to include the ```json)>>"""

In [ ]:
# 第一次 .format 把 {destinations} 替换成真实的候选列表文本，剩下 {input} 占位符留给后面用
router_template = MULTI_PROMPT_ROUTER_TEMPLATE.format(
    destinations=destinations_str
)
# RouterOutputParser 负责把 LLM 输出的 markdown JSON 解析成 {"destination": ..., "next_inputs": ...}
router_prompt = PromptTemplate(
    template=router_template,
    input_variables=["input"],
    output_parser=RouterOutputParser(),
)

# LLMRouterChain：根据 router_prompt 的解析结果，决定接下来把请求路由到哪个 destination chain
router_chain = LLMRouterChain.from_llm(llm, router_prompt)

In [ ]:
# MultiPromptChain 把 router_chain、destination_chains、default_chain 组合成一个完整的"智能路由"链：
# 先用 router_chain 判断问题类型，再把问题转发给对应的专家 chain（或 default_chain 兜底）
chain = MultiPromptChain(router_chain=router_chain,
                         destination_chains=destination_chains,
                         default_chain=default_chain, verbose=True
                        )

In [ ]:
# 预期：路由到 physics 专家链
chain.run("What is black body radiation?")

In [ ]:
# 预期：路由到 math 专家链
chain.run("what is 2 + 2")

In [ ]:
# 这个问题跟物理/数学/历史/计算机科学都不完全贴合，用来观察路由链的判断（可能落到 DEFAULT 或某个相近学科）
chain.run("Why does every cell in our body contain DNA?")